In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
from transformers import ViTFeatureExtractor, ViTForImageClassification
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report

In [3]:
train_dir = '/content/drive/My Drive/BE Project/chilli_plant_images/train'
val_dir = '/content/drive/My Drive/BE Project/chilli_plant_images/val'
test_dir = '/content/drive/My Drive/BE Project/chilli_plant_images/test'

In [4]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to ViT input size
    transforms.RandomHorizontalFlip(),  # Augmentation for training
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets and data loaders
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model initialization
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k', num_labels=5)

# Move model to appropriate device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Optimizer and loss function
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)  # Adjust learning rate as needed
loss_fn = nn.CrossEntropyLoss()

# Training loop
num_epochs = 5  # Adjust as needed
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images).logits
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

    # Validation loop (optional)
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            images, labels = batch
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images).logits
            # Calculate and print validation metrics
            # ...

# Evaluation on test dataset
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        images, labels = batch
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images).logits
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate and print test metrics
accuracy = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {accuracy}")
print(classification_report(all_labels, all_preds))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Test Accuracy: 0.94
              precision    recall  f1-score   support

           0       1.00      0.80      0.89        10
           1       0.77      1.00      0.87        10
           2       1.00      1.00      1.00        10
           3       1.00      1.00      1.00        10
           4       1.00      0.90      0.95        10

    accuracy                           0.94        50
   macro avg       0.95      0.94      0.94        50
weighted avg       0.95      0.94      0.94        50



In [ ]:
from PIL import Image

model.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Function to predict the class of a single image (using the currently trained model)
def predict_image(image_path):
    image = Image.open(image_path)
    input_tensor = preprocess(image).unsqueeze(0)  # Add batch dimension
    input_tensor = input_tensor.to(device)  # Move to device (GPU if available)

    with torch.no_grad():
        outputs = model(input_tensor).logits  # Use the 'model' variable directly
        probabilities = torch.nn.functional.softmax(outputs, dim=1)  # Get probabilities
        confidence_score, predicted_class = torch.max(probabilities, 1)

    class_names = train_dataset.classes  # Get class names from your training dataset
    predicted_class_name = class_names[predicted_class.item()]

    return predicted_class_name, confidence_score.item()

# ... (After the training loop) ...

# Example usage:
image_path = 'image.jpg'
predicted_class_name, confidence_score = predict_image(image_path)
print(f"Predicted class: {predicted_class_name}, Confidence Score: {confidence_score:.2f}")

OSError: image file is truncated (34 bytes not processed)

In [5]:
torch.save(model.state_dict(), 'disease_prediction_model_torch.pth')